# NB6 · Web'den çağrılabilen demo uygulaması
### A web-callable demo application

**Üretken Yapay Zekâ Araçları ile Klinik Karar Destek Sistemleri Geliştirilmesi**  
Teknoloji ve Yapay Zekâ Okuryazarlığı Eğitimi · Akdeniz Üniversitesi · 18 Eylül 2026

Prof. Dr. Utku Köse · Süleyman Demirel Üniversitesi, Bilgisayar Mühendisliği Bölümü  
Yapay Zekâ Uygulama ve Araştırma Merkezi (YAZEM) Müdürü · utkukose@sdu.edu.tr  
[ORCID 0000-0002-9652-6415](https://orcid.org/0000-0002-9652-6415) · [utkukose.com](https://www.utkukose.com) · [github.com/utkukose](https://github.com/utkukose)

---

Bu defter, önceki beş defterde kurulan sistemi tarayıcıdan kullanılabilen bir arayüze
bağlamaktadır. Gradio, Colab içinde geçici bir genel bağlantı üretmekte ve uygulama o
bağlantı üzerinden çalıştırılabilmektedir.

Arayüzde gösterilen şey bilinçli olarak seçilmiştir. Ekranda tek bir olasılık değil,
kararın kendisi, bariyerlerin durumu ve katkı dökümü birlikte yer almaktadır. Derste
anlatılan ayrım buydu: Karar destek sisteminin ürünü bir teşhis değil, gerekçesiyle
birlikte sunulan bir dikkat tahsisidir.


## 0. Kurulum · Setup


In [ ]:
!pip -q install pandas numpy scikit-learn matplotlib

import urllib.request

REPO = 'https://raw.githubusercontent.com/utkukose/cdss-genai-NB-lecture/main'
for module in ['mimic_web.py', 'evaluate.py', 'explain.py', 'safety.py', 'pipeline.py']:
    urllib.request.urlretrieve(f'{REPO}/workshop/{module}', module)

import numpy as np
import pandas as pd
!pip -q install gradio

import evaluate as ev
import explain as ex
import safety as sf
import pipeline as pl

state = pl.prepare(verbose=False)
model, train, test = state['model'], state['train'], state['test']
features, y_test = state['features'], state['y_test']
probabilities = state['probabilities']

threshold = ev.threshold_for_sensitivity(y_test, probabilities, target=0.80)
band = sf.choose_band(y_test, probabilities, threshold, max_abstain=0.20)
policy = sf.AbstentionPolicy(threshold, band['band'])
drift = sf.DriftDetector().fit(train[features])
validator = sf.PhysiologicalValidator().fit(train[features])
guarded = sf.GuardedModel(model, policy, drift, validator)

print(f'Sistem hazır. Eşik {threshold:.3f}, çekimserlik bandı ±{band["band"]:.3f}.')


## 1. Arayüzde gösterilecek öznitelikler

Otuzdan fazla öznitelik için kaydırıcı koymak arayüzü kullanılamaz hâle getirmektedir.
Permütasyon önemine göre ilk altı sayısal öznitelik seçilmekte, kalanlar eğitim
kümesinin ortancasıyla doldurulmaktadır.

Bu seçimin bir maliyeti bulunmaktadır ve arayüzde gösterilmektedir: Kullanıcı
görmediği özniteliklerin varsayılan değerlerde tutulduğunu bilmelidir.


In [ ]:
importance = ex.permutation_global(model, test[features], y_test, n_repeats=10, top=40)
numeric = set(train[features].select_dtypes(include='number').columns)

SLIDERS = [f for f in importance['feature'] if f in numeric][:6]
CATEGORICAL = [f for f in features if f not in numeric][:2]

defaults = train[features].median(numeric_only=True)

print('Kaydırıcılar:', SLIDERS)
print('Açılır listeler:', CATEGORICAL)


## 2. Tahmin işlevi · The prediction function


In [ ]:
def build_row(values: dict) -> pd.DataFrame:
    row = {}
    for column in features:
        if column in values:
            row[column] = values[column]
        elif column in numeric:
            row[column] = float(defaults.get(column, 0.0))
        else:
            mode = train[column].mode()
            row[column] = mode.iloc[0] if len(mode) else None
    return pd.DataFrame([row])[features]


# Demografik öznitelikler klinik gerekçe olarak sunulamaz. Katkı tablosunda
# göründüklerinde ayrı bir uyarı satırına taşınmaktadırlar; bu bir adalet denetimi
# bulgusudur, klinisyene gösterilecek bir sebep değildir.
PROTECTED = ['race', 'gender', 'insurance', 'marital_status', 'language']


def is_protected(feature_name: str) -> bool:
    return any(p in feature_name for p in PROTECTED)


ACIKLAMA = {
    'alert': 'UYARI · bu hastaya öncelik veriniz',
    'no alert': 'uyarı yok',
    sf.ABSTAIN: 'ÇEKİMSER · yeterli güven yok, klinisyen değerlendirmesi gerekli',
    'rejected, implausible input': 'REDDEDİLDİ · girdi fizyolojik olarak mümkün değil',
}


def cevir(decision: str) -> str:
    if decision in ACIKLAMA:
        return ACIKLAMA[decision]
    if decision.startswith('withheld'):
        return 'DEVREDİLDİ · hasta eğitim popülasyonuna benzemiyor, karar klinisyende'
    return decision


def predict(*inputs):
    values = dict(zip(SLIDERS + CATEGORICAL, inputs))
    row = build_row(values)
    result = guarded.predict_one(row)
    probability = result['probability']

    lines = [f'### {cevir(result["decision"])}']
    if probability is not None:
        lines.append(f'Model olasılığı: **{probability:.3f}**  ·  karar eşiği {threshold:.3f}')

    if result['reason']:
        lines.append('')
        if isinstance(result['reason'], list):
            for item in result['reason'][:3]:
                parts = ', '.join(f'{k}: {v}' for k, v in item.items())
                lines.append(f'- {parts}')
        else:
            lines.append(f'- {result["reason"]}')

    hidden = len(features) - len(SLIDERS) - len(CATEGORICAL)
    lines += ['', f'Arayüzde gösterilmeyen {hidden} öznitelik eğitim kümesinin ortanca '
                  'değerinde tutulmaktadır.']

    if probability is None:
        return '\n'.join(lines), pd.DataFrame()

    contributions = ex.linear_contributions(model, row, top=20)
    protected = contributions[contributions['feature'].map(is_protected)]
    clinical = contributions[~contributions['feature'].map(is_protected)].head(8)

    if len(protected) and protected['contribution_to_log_odds'].abs().max() > 0.2:
        worst = protected.iloc[0]
        lines += ['', f'**Adalet denetimi gerekli.** `{worst["feature"]}` bu tahmine '
                      f'{worst["contribution_to_log_odds"]:+.2f} log odds katkı veriyor. '
                      'Demografik bir öznitelik klinik gerekçe olarak sunulamaz.']

    table = clinical[['feature', 'contribution_to_log_odds']].round(3)
    return '\n'.join(lines), table


print(predict(*[float(defaults.get(s, 0)) for s in SLIDERS],
               *[train[c].mode().iloc[0] for c in CATEGORICAL])[0])


## 3. Arayüz · The interface

`share=True` parametresi Colab içinde geçici bir genel bağlantı üretmektedir. Bağlantı
yaklaşık bir hafta geçerlidir ve oturum kapandığında çalışmayı durdurmaktadır.

Bağlantıyı paylaşırken arayüzün üzerindeki uyarı metninin göründüğünden emin olunuz.


In [ ]:
import gradio as gr

UYARI = (
    '**Bu bir öğretim prototipidir, doğrulanmış bir klinik araç değildir.** '
    'MIMIC-IV demo verisiyle, yüz hastalık bir kümede eğitilmiştir. Gerçek hasta '
    'kararlarında kullanılamaz.'
)

with gr.Blocks(title='Yoğun bakımda uzamış kalış riski') as demo:
    gr.Markdown('# Yoğun bakımda uzamış kalış riski')
    gr.Markdown(
        'Kabulden altı saat sonra, yatışın üç günü aşma riski.  \n'
        'Prof. Dr. Utku Köse · Süleyman Demirel Üniversitesi · YAZEM'
    )
    gr.Markdown(UYARI)

    controls = []
    with gr.Row():
        with gr.Column():
            for name in SLIDERS:
                low = float(train[name].quantile(0.01))
                high = float(train[name].quantile(0.99))
                start = float(defaults.get(name, (low + high) / 2))
                controls.append(gr.Slider(low, high, value=start, label=name))
            for name in CATEGORICAL:
                choices = [c for c in train[name].dropna().unique().tolist()]
                start = train[name].mode().iloc[0] if len(train[name].mode()) else None
                controls.append(gr.Dropdown(choices, value=start, label=name))
            run = gr.Button('Değerlendir', variant='primary')

        with gr.Column():
            verdict = gr.Markdown()
            table = gr.Dataframe(label='Log odds katkıları')

    run.click(predict, inputs=controls, outputs=[verdict, table])

demo.launch(share=True, debug=False)


## 4. Arayüzde ne denemeli · What to try in the interface

Beş deneme yapınız ve her birinde sistemin ne söylediğine bakınız.

**Tipik bir hasta.** Varsayılan değerlerle çalıştırınız. Karar ne, olasılık kaç?

**Eşiğin hemen etrafında bir hasta.** Kaydırıcıları yavaşça hareket ettirerek olasılığı
eşiğe yaklaştırınız. Sistem çekimser kalmaya başladığı noktayı bulunuz. Klinik pratikte
bu bant, kimin listeye alınıp kimin klinisyene bırakılacağını belirlemektedir.

**Dağılım dışı bir hasta.** Birkaç kaydırıcıyı uçlara çekiniz. Sistem tahmin üretmeyi
bırakıp devretme mesajı vermelidir.

**Aynı olasılık, farklı gerekçe.** Farklı kaydırıcı kombinasyonlarıyla benzer bir
olasılığa ulaşınız ve katkı tablosunun değiştiğini görünüz. İki hasta aynı riski
taşıyabilir, ama klinisyenin yapması gereken şey farklıdır. Bu, ekranda yalnızca bir
sayı göstermenin neden yetersiz olduğunu doğrudan göstermektedir.

**Cinsiyeti değiştiriniz, başka hiçbir şeye dokunmayınız.** Olasılık değişiyorsa ve
adalet denetimi uyarısı çıkıyorsa, modeliniz demografik bir öznitelikten öğrenmiş
demektir. Bu bulgu arayüzde bilerek ayrı bir satırda gösterilmektedir: Demografik bir
öznitelik, bir klinisyene sunulacak gerekçe listesine karıştırılamaz. Katkının kendisi
bir adalet denetimi bulgusudur ve modelin devreye alınmasını engelleyebilecek türdendir.


## 5. Bundan sonrası · What comes next

Prototip çalışmaktadır ve bu, sistemin hazır olduğu anlamına gelmemektedir. Derste
anlatılan zincirin yalnızca ilk halkaları tamamlanmıştır.

Eksik olanlar şunlardır: Başka bir merkezde dış doğrulama, klinisyen kaynaklı
fizyolojik sınırlar, iş akışına yerleştirme çalışması, eşiğin klinik ekip tarafından
onaylanması, düzenleyici sınıflandırma, kişisel veri değerlendirmesi ve devreye alma
sonrası başarım izleme.

Bu liste, atölyede iki saatte yapılan işin klinik bir ürün yolculuğunun neresinde
durduğunu göstermektedir. Kod yazma maliyeti düşmüştür; geriye kalan her şey
düşmemiştir.


---

### Uyarı

Bu defterde üretilen hiçbir model doğrulanmış bir klinik araç değildir. MIMIC-IV demo
verisi tek bir Amerikan hastanesinden gelmektedir ve Türkiye'deki bir yoğun bakım
popülasyonunu temsil etmemektedir. Buradaki çıktılar öğretim amaçlıdır.
